In [ ]:
# Notebook: ResNet50 Training for Pneumonia Detection

# Importations
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import sys
sys.path.append('..')  # Ajouter le répertoire parent au chemin
from src.model.resnet50_model import create_resnet50_model, fine_tune_resnet50

# 1. Configuration
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
EPOCHS_FINE_TUNE = 10

# 2. Préparation des données
# Générateurs de données avec augmentation pour l'entraînement
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Générateurs sans augmentation pour validation et test
valid_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

# Charger les données à partir des répertoires
train_generator = train_datagen.flow_from_directory(
    '../data/processed/train',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_generator = valid_datagen.flow_from_directory(
    '../data/processed/val',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    '../data/processed/test',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# 3. Créer le modèle ResNet50
model = create_resnet50_model(input_shape=(224, 224, 3))
model.summary()

# 4. Callbacks pour l'entraînement
checkpoint_dir = '../models/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        os.path.join(checkpoint_dir, 'resnet50_best.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6
    )
]

# 5. Entraînement initial (feature extraction)
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator,
    callbacks=callbacks
)

# 6. Sauvegarder le modèle
model.save('../models/resnet50_base.h5')

# 7. Visualiser les résultats d'entraînement
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Précision du modèle')
plt.ylabel('Précision')
plt.xlabel('Epoch')
plt.legend(['Entraînement', 'Validation'], loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Perte du modèle')
plt.ylabel('Perte')
plt.xlabel('Epoch')
plt.legend(['Entraînement', 'Validation'], loc='upper right')

plt.tight_layout()
plt.show()

# 8. Fine-tuning du modèle
fine_tuned_model = fine_tune_resnet50(model)

# 9. Entraînement avec fine-tuning
history_fine_tune = fine_tuned_model.fit(
    train_generator,
    epochs=EPOCHS_FINE_TUNE,
    validation_data=valid_generator,
    callbacks=callbacks
)

# 10. Sauvegarder le modèle final
fine_tuned_model.save('../models/resnet50_finetune.h5')

# 11. Évaluation sur l'ensemble de test
test_results = fine_tuned_model.evaluate(test_generator)
print(f"Résultats du test - Perte: {test_results[0]:.4f}, Précision: {test_results[1]:.4f}")
print(f"Precision: {test_results[2]:.4f}, Recall: {test_results[3]:.4f}, AUC: {test_results[4]:.4f}")

# 12. Prédictions sur l'ensemble de test
predictions = fine_tuned_model.predict(test_generator)
predictions_binary = (predictions > 0.5).astype(int)

# 13. Matrice de confusion
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

true_labels = test_generator.classes
cm = confusion_matrix(true_labels, predictions_binary)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Pneumonie'],
            yticklabels=['Normal', 'Pneumonie'])
plt.xlabel('Prédiction')
plt.ylabel('Vérité terrain')
plt.title('Matrice de confusion')
plt.show()

# 14. Rapport de classification
print("Rapport de classification:")
print(classification_report(true_labels, predictions_binary, 
                           target_names=['Normal', 'Pneumonie']))